In [8]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [9]:
# read data

df = pd.read_csv('src/liver_disorder.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col
y = y.replace({1: 0, 2: 1})

In [10]:

# using state=42 makes result reproducable. remove for random
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape 18-12
print(X_train.shape)
print(X_test.shape)

(207, 6)
(138, 6)


In [11]:
y_train = y_train.values
y_test = y_test.values

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))


print(y_train.shape)
print(X_train.shape)

(207,)
(207, 7)


In [12]:
# need a weight for all inputs BP Cholesterol Age Pregnant (w1,w2,w3,w4)
# need w0 (bias)
# score = w1*BP + w2*Cholesterol + w3*Age + w4*Pregnant + b

prev_loss = 0
current_loss = 0
threshold = 1e-4

fail_safe = 1000
iterations = 0
learning_rate = 0.01

weights = np.zeros(X_train.shape[1])

while iterations < fail_safe:
    # 1. predict
    z = np.dot(X_train, weights)
    y_pred = sigmoid(z)

    # 2. calculate loss
    m = len(y_train)
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    current_loss = -(1/m) * np.sum(
        y_train * np.log(y_pred) + (1 - y_train) * np.log(1 - y_pred)
    )

    # 3. stop if converged
    if iterations > 0 and abs(current_loss - prev_loss) < threshold:
        print(f"Iterations: {iterations} | current_loss: {current_loss}")
        break

    # 4. check error
    error = y_pred - y_train

    # 5. adjust weight
    gradient = np.dot(X_train.T, error) / m
    weights = weights - learning_rate * gradient

    # 6. update for next loop
    prev_loss = current_loss
    iterations += 1

    print(f"Iterations: {iterations} | current_loss: {current_loss}")


Iterations: 1 | current_loss: 0.6931471805599452
Iterations: 2 | current_loss: 0.692930739937652
Iterations: 3 | current_loss: 0.6927153505418432
Iterations: 4 | current_loss: 0.6925010047464807
Iterations: 5 | current_loss: 0.6922876949916048
Iterations: 6 | current_loss: 0.6920754137828115
Iterations: 7 | current_loss: 0.6918641536907278
Iterations: 8 | current_loss: 0.6916539073504916
Iterations: 9 | current_loss: 0.6914446674612272
Iterations: 10 | current_loss: 0.6912364267855269
Iterations: 11 | current_loss: 0.69102917814893
Iterations: 12 | current_loss: 0.6908229144394047
Iterations: 13 | current_loss: 0.6906176286068301
Iterations: 14 | current_loss: 0.690413313662481
Iterations: 15 | current_loss: 0.6902099626785128
Iterations: 16 | current_loss: 0.6900075687874484
Iterations: 17 | current_loss: 0.6898061251816675
Iterations: 18 | current_loss: 0.6896056251128964
Iterations: 19 | current_loss: 0.6894060618917002
Iterations: 20 | current_loss: 0.6892074288869772
Iterations: 2

In [13]:
# run the algo again but on test data
z_test = np.dot(X_test, weights)
y_prob = sigmoid(z_test)

# change probability into class label. Threshold is 0.5
y_pred = (y_prob >= 0.5).astype(int)

# compare accuracy
accuracy = np.mean(y_pred == y_test)

print(f"Accurary: {accuracy}")

Accurary: 0.6304347826086957


In [14]:
print(f"""
weights: 
    b: {weights[0]}
    BP: {weights[1]}
    Cholesterol: {weights[2]}
    Age: {weights[3]}
    Pregnant: {weights[4]}

""")


weights: 
    b: 0.14044972999004993
    BP: -0.0607854521338804
    Cholesterol: -0.06254977085955571
    Age: -0.051314366432831826
    Pregnant: 0.11084519493396504


